# Modelo final — objetivo CTC en **palabras**

Entrenamiento definitivo de `best_word_eeg`: **una** configuracion, **un**
entrenamiento de hasta 80 epocas, decodificacion con modelo de lenguaje y las
metricas de la memoria. Sin barridos, sin ablaciones, sin registro de
experimentos: eso es lo que hace `owsm-eeg-v12`.

Vocabulario cerrado de las palabras de train. El lexico esta metido en la capa
de salida: el modelo no puede escribir una no-palabra. Tiene un suelo duro de
WER del 6.2% por las palabras OOV de validacion, medido en la auditoria. Fue el
mejor del barrido.

**Referencia del barrido para esta unidad:** CER 0.222 · WER 0.282  (40 epocas)

## Que hace, en orden

| seccion | que |
|---|---|
| 0 | instalacion (una vez, y reiniciar) |
| 1–5 | configuracion, datos, modelo |
| 6 | preparacion y verificacion del objetivo CTC |
| 7 | entrenamiento (80 epocas, early stopping) |
| 8 | metricas CTC greedy sobre validacion completa |
| 9 | decodificacion con n-grama y oraculo |
| 10 | tabla final y curva |

## La receta

Es la del bloque A del barrido, que gano a todas las demas: encoder
**inicializado al azar** (no con OWSM), frontend profundo con submuestreo 2,
capa por sesion, InterCTC en la capa 2, `ctc_weight=1.0` (sin decoder de
atencion) y `enc_lr_scale=0.05`.

Ese ultimo valor **no** esta protegiendo pesos preentrenados —no los hay—: es
estabilidad de optimizacion de esta arquitectura. Con `enc_lr_scale=1.0` el
entrenamiento diverge tanto con init aleatorio como con OWSM. Cuatro runs del
barrido lo demuestran (`A1randLr1`, `H3owsmEls02`, `D2bUlt2Lr1`, `D3bUlt4Lr1`).

**Coste:** ~10 h y ~18 unidades de computo en una L4 si agota las 80 epocas.
Con `patience=20` desde la epoca 40 probablemente pare antes.

## 0. Instalacion

**Ejecuta esta celda, reinicia el entorno y sigue desde la seccion 1.**

Todo se instala aqui (ESPnet, kenlm, pyctcdecode) porque instalar a mitad de
sesion cambia numpy en disco bajo un kernel que ya lo tiene cargado, y a partir
de ahi scipy falla con mensajes que no senalan la causa. Dos trampas conocidas:
`pyctcdecode` declara `numpy<2.0.0` (se instala con `--no-deps`) y
`espnet → librosa → numba` pone tope superior a numpy (se fija con un fichero
de *constraints*).

In [ ]:
!nvidia-smi

# ══════════════════════════════════════════════════════════════════════
# INSTALACION UNICA (v6.2). Al terminar: REINICIA EL ENTORNO.
# ══════════════════════════════════════════════════════════════════════
import importlib.util, subprocess, sys, os, textwrap

def sh(cmd, titulo=None, critico=True):
    """Ejecuta y solo escupe la salida si falla (los logs de apt son ruido)."""
    if titulo:
        print(f"  {titulo} ...", end=" ", flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = r.returncode == 0
    if titulo:
        print("ok" if ok else "FALLO", flush=True)
    if not ok:
        print(textwrap.indent((r.stdout + r.stderr)[-3000:], "    "))
        if critico:
            raise RuntimeError(f"Fallo: {cmd}")
    return ok

def probar(codigo):
    """Importa en un proceso NUEVO. Es la unica prueba que vale: este kernel
    ya tiene modulos cargados en memoria y puede dar un falso positivo."""
    r = subprocess.run([sys.executable, "-c", codigo],
                       capture_output=True, text=True)
    lineas = r.stderr.strip().splitlines()
    return r.returncode == 0, r.stdout.strip(), (lineas[-1] if lineas else "")

# ── 0. Cual es el numpy bueno de esta VM ──────────────────────────────
# El scipy que trae Colab esta compilado contra SU numpy. Ese es el que hay
# que preservar. Si ya viene roto de un intento anterior, se restaura.
ok, ver_numpy, err = probar("import scipy.special, numpy; print(numpy.__version__)")
if not ok:
    print(f"  numpy actual incompatible con scipy ({err})")
    sh(f"{sys.executable} -m pip install -q -U numpy numba", "  restaurando numpy y numba")
    ok, ver_numpy, err = probar("import scipy.special, numpy; print(numpy.__version__)")
    if not ok:
        raise RuntimeError(f"numpy y scipy siguen sin entenderse: {err}\n"
                           "Entorno de ejecucion > Desconectar y eliminar el entorno, "
                           "y empieza en una VM limpia.")
print(f"  numpy de referencia: {ver_numpy} (se fija durante toda la instalacion)")

CONS = "/content/constraints.txt"
with open(CONS, "w") as f:
    f.write(f"numpy=={ver_numpy}\n")

PIP = f"{sys.executable} -m pip install -q -c {CONS}"

# ── 1. ESPnet ─────────────────────────────────────────────────────────
# El -c es lo que impide que numba arrastre numpy hacia abajo. Truco para
# sesiones siguientes: cachea las ruedas en Drive una sola vez con
#   !pip download -q espnet espnet_model_zoo loralib h5py -d /content/drive/MyDrive/TFG/wheels
# y luego instala offline en ~20 s con
#   !pip install -q --no-index --find-links=/content/drive/MyDrive/TFG/wheels espnet espnet_model_zoo loralib h5py
sh(f"{PIP} espnet espnet_model_zoo loralib h5py", "espnet + h5py + loralib")

# ── 2. kenlm (compilado desde fuente) ─────────────────────────────────
# Por que no basta con 'pip install kenlm': el sdist trae python/kenlm.cpp
# PREGENERADO con Cython 0.29.35, que no compila contra las cabeceras de
# Python 3.13 (las que usa Colab). De ahi el "Building wheel for kenlm ...
# error" sin causa visible. Se regenera el binding con Cython 3 y listo.
if importlib.util.find_spec("kenlm") is None:
    sh("apt-get install -y -qq build-essential cmake libboost-all-dev "
       "zlib1g-dev libbz2-dev liblzma-dev", "dependencias de compilacion")
    sh(f"{PIP} 'cython>=3'", "cython 3")
    sh("rm -rf /content/kenlm && "
       "git clone -q --depth 1 https://github.com/kpu/kenlm.git /content/kenlm",
       "clonando kenlm")
    sh("cd /content/kenlm && cython --cplus python/kenlm.pyx -o python/kenlm.cpp",
       "regenerando el binding con Cython 3")
    sh(f"cd /content/kenlm && {PIP} --no-build-isolation --no-deps .",
       "compilando kenlm (~2 min)")
    # lmplz: el estimador de Kneser-Ney. El kenlm de pip NO lo trae, y por eso
    # la version antigua del notebook generaba el ARPA con add-k en Python
    # puro. Si falla no es critico: hay respaldo (ver seccion 13).
    sh("cd /content/kenlm && mkdir -p build && cd build && "
       "cmake .. -DCMAKE_BUILD_TYPE=Release && make -j$(nproc) lmplz",
       "compilando lmplz (Kneser-Ney)", critico=False)
else:
    print("  kenlm ya instalado, se salta la compilacion")

# ── 3. pyctcdecode SIN sus dependencias ───────────────────────────────
# pygtrie es la unica dependencia real en ejecucion (la usa el argumento
# unigrams= de build_ctcdecoder); hypothesis es solo de tests.
sh(f"{PIP} --no-deps pyctcdecode pygtrie", "pyctcdecode (sin deps)")

# ── 4. VERIFICACION EN PROCESO LIMPIO ─────────────────────────────────
# Aqui es donde se detecta el problema, no tres celdas mas abajo.
print("\n  verificando (importa espnet, tarda ~30 s) ...", flush=True)
ok, salida, err = probar(textwrap.dedent("""
    import numpy, scipy.special, torch
    import espnetez, kenlm, pyctcdecode
    from espnet2.bin.s2t_inference import Speech2Text
    print(f"numpy={numpy.__version__} torch={torch.__version__} "
          f"cuda={torch.cuda.is_available()}")
"""))

print()
if not ok:
    print("=" * 70)
    print("LA INSTALACION NO HA QUEDADO BIEN. No reinicies todavia.")
    print("=" * 70)
    print(err)
    print(f"\nnumpy instalado ahora: ", end="")
    subprocess.run([sys.executable, "-c",
                    "import numpy;print(numpy.__version__)"])
    print(f"numpy que deberia haber: {ver_numpy}")
    raise RuntimeError("Ver el error de arriba.")

print(f"  {salida}")
print(f"  lmplz: {'si' if os.path.exists('/content/kenlm/build/bin/lmplz') else 'no (se usara el ARPA add-k en Python puro)'}")
print()
print("=" * 70)
print("  TODO CORRECTO. Ahora: Entorno de ejecucion > Reiniciar sesion,")
print("  y sigue desde la celda 1.")
print("=" * 70)

## 1. Configuracion

Un solo diccionario. `CFG` es la receta ganadora del barrido con el objetivo
fijado a `word` y el presupuesto subido a 80 epocas.

In [ ]:
# ══ RUTAS ══════════════════════════════════════════════════════════
DRIVE_ROOT  = "/content/drive/MyDrive/TFG"
DRIVE_DATA  = f"{DRIVE_ROOT}/DatosEEG_crudos/hdf5_data_final"
DATA_ROOT   = "/content/data/hdf5_data_final"     # cache en el disco local
OUTPUT_DIR  = "/content/exp"                      # checkpoints en local
RESULTS_DIR = f"{DRIVE_ROOT}/resultados_finales"

UNIDAD = "word"
TAG    = f"best_{UNIDAD}"

# ══ MODELO Y DATOS ═════════════════════════════════════════════════
FINETUNE_MODEL = "espnet/owsm_v3.1_ebf_base"
LANGUAGE       = "eng"
LORA_TARGET    = ["w_1", "w_2", "merge_proj", "linear_q", "linear_k",
                  "linear_v", "linear_out"]
PRELOAD_RAM    = False    # lectura perezosa del HDF5: seguro con 45 sesiones
SOLO_SESIONES_COMPLETAS = False

# ══ EVALUACION ═════════════════════════════════════════════════════
EVAL_N_GREEDY = None   # None = validacion completa (~1.426 trials)
EVAL_N_BEAM   = 200    # sin uso aqui: con ctc_weight=1.0 no hay decoder
BEAM_SIZE     = 5
EVAL_GREEDY_CTC = True
GUARDAR_CKPT  = True   # el .pth a Drive. Con False muere con la VM y no se
                       # puede volver a evaluar ni pasar el LM.
SKIP_IF_DONE = REPETIR_FALLIDOS = False    # aqui no hay barrido que saltar

# ══ LA RECETA ══════════════════════════════════════════════════════
# Bloque A del barrido. Lo unico que cambia entre los tres notebooks finales
# es objetivo_ctc.
CFG = dict(
    nombre = TAG,
    objetivo_ctc = UNIDAD,
    bpe_vocab    = 256,

    n_sessions = 45,
    max_epoch  = 80,
    warmup     = 1500,
    patience   = 20,               # para si 20 epocas seguidas no mejora cer_ctc
    patience_start_epoch = 40,     # ...pero no antes de la 40: H1 seguia bajando ahi
    criterio   = "cer_ctc",        # NO "loss": con ctc_weight=1 la loss no es la metrica

    # ── encoder: al azar, no OWSM. Es el resultado central del TFG ──
    init_encoder = "random",
    n_capas_encoder = None,        # las 6 de OWSM. Con 3 el CER solo sube 0.017
    unfreeze_encoder = True, unfreeze_ultimas = None, unfreeze_n_layers = None,
    usar_lora = False,

    # ── frontend ──
    frontend = "deep", deep_dim = 512, deep_bloques = 2,
    deep_kernel = 5, deep_dropout = 0.1, subsample = 2,
    capa_sesion = True,            # la caracteristica del baseline de B2T'25
    norm = "ninguna",              # los HDF5 ya vienen z-scoreados por bloque

    # ── perdida ──
    ctc_weight = 1.0,              # sin decoder de atencion
    ctc_weight_decode = None,
    interctc_weight = 0.3, interctc_layers = (2,),

    # ── optimizacion ──
    lr = 1e-3,
    enc_lr_scale = 0.05,           # estabilidad, NO proteccion de pesos: con 1.0
                                   # diverge tambien con init aleatorio
    optim = "adamw", weight_decay = 1e-2, grad_clip = 100.0,
    batch_type = "sorted", batch_size = 16, batch_bins = 6_000_000, accum_grad = 2,
    num_workers = 0,               # 0 = sin fork; HDF5 no es fork-safe
    seed = 2024,

    # ── aumento (solo en train) ──
    aug = True, aug_ruido = 0.5, aug_offset = 0.2,
    aug_n_tiempo = 2, aug_max_tiempo = 0.05,
    aug_n_canales = 2, aug_max_canales = 20,
    suavizado = 0.0,

    # ── sin holdout: se evalua sobre la validacion de las mismas sesiones ──
    holdout_sesiones = 0, holdout_capa_sesion = "identidad",
    val_igual_train = False,
)
DEFAULTS = CFG                     # el codigo heredado lee DEFAULTS

# La semilla es la 2024 por coherencia con el resto del barrido. NO se elige la
# mejor de las tres (7 -> 0.297, 1234 -> 0.314, 2024 -> 0.324): elegir semilla
# mirando validacion y luego reportar validacion seria contaminar la cifra.
# La dispersion entre semillas (+-0.011 de CER) se reporta al lado del numero.

print(f"Objetivo: {UNIDAD} · {CFG['max_epoch']} epocas · semilla {CFG['seed']}")

## 2. Drive y datos

Los HDF5 se copian de Drive al disco local de la VM. Entrenar leyendo
directamente de Drive es mucho mas lento y se cuelga con lecturas aleatorias.
La copia verifica tamano y que `h5py` pueda abrir el fichero: comprobar solo
`os.path.exists` deja pasar copias truncadas.

In [ ]:
import os, time, shutil, glob

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# Localizar la carpeta de datos aunque este en otro sitio dentro de DRIVE_ROOT
if not os.path.isdir(DRIVE_DATA):
    print(f"No esta en {DRIVE_DATA}; buscando 'hdf5_data_final' bajo {DRIVE_ROOT} ...")
    cands = glob.glob(f"{DRIVE_ROOT}/**/hdf5_data_final", recursive=True)
    if cands:
        DRIVE_DATA = cands[0]
        print("Encontrada:", DRIVE_DATA)
    else:
        raise FileNotFoundError(
            f"No encuentro la carpeta hdf5_data_final dentro de {DRIVE_ROOT}.\n"
            f"Ajusta DRIVE_ROOT / DRIVE_DATA en el panel de control."
        )

# Inventario: que sesiones hay y cuales tienen train Y val
inventario, sin_val, bytes_tot = [], [], 0
for s in sorted(os.listdir(DRIVE_DATA)):
    d = os.path.join(DRIVE_DATA, s)
    if not os.path.isdir(d):
        continue
    tr = os.path.join(d, "data_train.hdf5")
    va = os.path.join(d, "data_val.hdf5")
    if not os.path.exists(tr):
        continue
    inventario.append(s)
    if not os.path.exists(va):
        sin_val.append(s)
    bytes_tot += os.path.getsize(tr) + (os.path.getsize(va) if os.path.exists(va) else 0)

print(f"\n{len(inventario)} sesiones con data_train.hdf5 · {bytes_tot/1e9:.1f} GB (train+val)")
if sin_val:
    print(f"{len(sin_val)} sin data_val.hdf5: {sin_val}")
    print("  -> con SOLO_SESIONES_COMPLETAS=True se ignoran (evita que train y val no cuadren)")
print("\nPrimeras sesiones:", inventario[:5])
print("Disco libre en la VM:")
os.system("df -h /content | tail -1")

## 3. Imports y utilidades

In [ ]:
# ══ GUARDA DE ABI numpy/scipy (v6.2) ══════════════════════════════════
# Sin esto, un numpy que no cuadra hace que `import espnetez` muera 40
# frames mas abajo dentro de scipy, con un mensaje que no menciona numpy:
#   ValueError: All ufuncs must have type `numpy.ufunc`   (numpy 1.x cargado)
#   ImportError: cannot import name '_slice' from 'numpy._core.umath'
#                                                        (numpy 2.1 con scipy 2.4)
# La comprobacion no es "que version es", porque 2.1.3 parece razonable y no
# lo es: es "importan scipy y numpy juntos, si o no".
try:
    import numpy as _np
    import scipy.special                # el primero que revienta si la ABI falla
except Exception as _e:
    try:
        import numpy as _np
        _v = _np.__version__
    except Exception:
        _v = "(no importa ni numpy)"
    raise RuntimeError(
        f"numpy {_v} y el scipy de esta VM no son compatibles.\n"
        f"  Error real: {type(_e).__name__}: {_e}\n"
        "\n"
        "  Dos causas posibles:\n"
        "   a) No reiniciaste el entorno despues de la celda 0. pip escribe en\n"
        "      disco, y sin reiniciar el proceso se queda con la version vieja\n"
        "      en memoria. Solucion: Reiniciar sesion y volver a la celda 1.\n"
        "   b) Algo degrado numpy despues (tipico: 'pip install pyctcdecode'\n"
        "      sin --no-deps, o espnet arrastrando una numba vieja que exige\n"
        "      numpy<2.2). Solucion: ejecuta la celda 0 entera; detecta el\n"
        "      numpy degradado y lo restaura antes de instalar nada.\n") from None

import string, re, glob, argparse, logging, gc, copy, json, math
import h5py
import numpy as np
import pandas as pd
import torch
import loralib

import espnetez as ez
from espnet2.bin.s2t_inference import Speech2Text
from espnet2.layers.create_adapter_fn import create_lora_adapter
from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
NGPU    = 1 if torch.cuda.is_available() else 0
USE_AMP = torch.cuda.is_available()

GPU_NAME = torch.cuda.get_device_name(0) if NGPU else "CPU"

# Tarifas aproximadas de compute units/hora (medidas en marzo de 2026; pueden variar)
TARIFAS = {"T4": 1.19, "L4": 1.71, "A100-SXM4-40GB": 5.40, "A100-SXM4-80GB": 7.52,
           "RTX PRO 6000": 8.71, "H100": 9.0}
def units_por_hora(nombre_gpu):
    for k, v in TARIFAS.items():
        if k.lower().replace("-", " ") in nombre_gpu.lower().replace("-", " "):
            return v
    return float("nan")
CU_H = units_por_hora(GPU_NAME)

print(f"Dispositivo: {DEVICE} · {GPU_NAME} · AMP: {USE_AMP}")
print(f"Coste estimado: {CU_H} compute units/hora (~${CU_H*0.10:.2f}/h)")


def normaliza(x, modo="trial"):
    """Normalizacion de la senal sEEG antes de entrar al modelo.

    Los HDF5 de B2T'25 ya vienen binneados a 20 ms y z-scoreados por bloque, asi que
    "ninguna" reproduce lo que hacen los trabajos publicados; "trial" vuelve a normalizar
    por trial y canal (util para comparar, pero borra la amplitud relativa entre trials).
    """
    x = x.astype(np.float32)
    if modo == "ninguna":
        return x
    return (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def decode_transcription(arr):
    arr = np.asarray(arr).ravel()
    return "".join(chr(int(x)) for x in arr if int(x) != 0)

## 4. Dataset

`SeeGDataset` guarda solo un indice `(fichero, clave)` y lee cada trial del
HDF5 cuando hace falta.

In [ ]:
class SeeGDataset(torch.utils.data.Dataset):
    """Lee trials del HDF5 bajo demanda. Si preload=True los cachea todos en RAM."""

    def __init__(self, index, preload=False):
        self.index = index          # [(ruta_hdf5, clave), ...]
        self.preload = preload
        self._files = {}
        self._pid = os.getpid()
        self._cache = None
        if preload:
            self._cache = [self._leer(i) for i in range(len(index))]
            self.cerrar()

    def _handle(self, path):
        if os.getpid() != self._pid:      # tras un fork, los handles del padre no valen
            self._files, self._pid = {}, os.getpid()
        if path not in self._files:
            self._files[path] = h5py.File(path, "r")
        return self._files[path]

    def cerrar(self):
        """Cierra los handles. IMPRESCINDIBLE antes de que el DataLoader haga fork:
        la libreria HDF5 no es fork-safe y heredar handles abiertos puede colgar el proceso."""
        for f in self._files.values():
            try: f.close()
            except Exception: pass
        self._files = {}

    def _leer(self, idx):
        path, key, sid = self.index[idx]
        t = self._handle(path)[key]
        d = {"input_features": t["input_features"][:],
             "text_raw": decode_transcription(t["transcription"][()]),
             "session_idx": sid}
        # (v6) secuencia de fonemas de ground truth. B2T'25 la incluye en cada
        # trial; el nombre exacto de la clave se autodetecta una sola vez.
        k = clave_fonemas(t)
        d["fon_ids"] = np.asarray(t[k][()], dtype=np.int64).ravel() if k else None
        return d

    def textos(self):
        """Solo las transcripciones, SIN leer input_features.

        Leer un trial entero cuesta ~1.6 MB; con 10.000 trials serian ~16 GB de
        disco para quedarse con una frase de cada uno. Esto abre cada fichero una
        sola vez y lee unicamente el dataset 'transcription'.
        """
        out = [None] * len(self.index)
        orden = sorted(range(len(self.index)), key=lambda i: self.index[i][0])
        actual, fh = None, None
        try:
            for i in orden:
                path, key, _ = self.index[i]
                if path != actual:
                    if fh is not None:
                        fh.close()
                    fh, actual = h5py.File(path, "r"), path
                out[i] = decode_transcription(fh[key]["transcription"][()])
        finally:
            if fh is not None:
                fh.close()
        return out

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        d = self._cache[idx] if self._cache is not None else self._leer(idx)
        text_lower = d["text_raw"].lower()
        return {
            "input_features": d["input_features"],
            "session_idx": d["session_idx"],
            "fon_ids": d.get("fon_ids"),
            "text":      f"<{LANGUAGE}><asr><notimestamps> {text_lower}",
            "text_prev": "<na>",
            "text_ctc":  remove_punctuation(text_lower),
            "text_raw":  d["text_raw"],
        }


_FON_KEY = "__sin_detectar__"

def clave_fonemas(grupo):
    """Autodetecta el nombre del dataset de fonemas dentro de un trial del HDF5.

    B2T'25 guarda la secuencia de fonemas de ground truth en cada trial, pero el
    nombre de la clave ha variado entre versiones del dataset. Se busca una sola
    vez y se cachea. Devuelve None si el HDF5 no la trae (p.ej. data_test.hdf5).
    """
    global _FON_KEY
    if _FON_KEY == "__sin_detectar__":
        candidatos = [k for k in grupo.keys()
                      if any(t in k.lower() for t in
                             ("phone", "seq_class", "seqclass", "phonem"))]
        _FON_KEY = candidatos[0] if candidatos else None
        print(f"  [fonemas] clave detectada en el HDF5: {_FON_KEY!r}"
              + ("" if _FON_KEY else "  <-- SIN ETIQUETAS DE FONEMA, objetivo_ctc='fon' no funcionara"))
    return _FON_KEY if (_FON_KEY and _FON_KEY in grupo) else None


def construir_indice(sesiones, split, solo=None):
    """Recorre las claves de cada HDF5 sin leer los datos.

    Cada entrada es (ruta, clave, indice_de_sesion). El indice de sesion es la
    posicion en la lista COMPLETA `sesiones`, no en el subconjunto: asi la capa
    por sesion tiene un slot por dia aunque ese dia solo aparezca en validacion.

    `solo` = subconjunto de nombres de sesion a incluir (None = todas).
    """
    index = []
    for sid, s in enumerate(sesiones):
        if solo is not None and s not in solo:
            continue
        path = os.path.join(DATA_ROOT, s, f"data_{split}.hdf5")
        if not os.path.exists(path):
            print(f"  [aviso] no encontrado: {path}")
            continue
        with h5py.File(path, "r") as f:
            index.extend((path, k, sid) for k in sorted(f.keys()))
    return index


def sesiones_disponibles():
    """Sesiones utilizables, listadas desde Drive (la fuente de la verdad).

    Con SOLO_SESIONES_COMPLETAS=False se incluyen tambien las sesiones que solo
    tienen data_train.hdf5: aportan datos de entrenamiento y simplemente no
    contribuyen a la validacion.
    """
    out = []
    for s in sorted(os.listdir(DRIVE_DATA)):
        d = os.path.join(DRIVE_DATA, s)
        if not os.path.isdir(d) or not os.path.exists(os.path.join(d, "data_train.hdf5")):
            continue
        if SOLO_SESIONES_COMPLETAS and not os.path.exists(os.path.join(d, "data_val.hdf5")):
            continue
        out.append(s)
    return out


def copiar_sesion(sesion):
    """Trae data_train/data_val de esa sesion al disco local de la VM.

    (v6) Comprueba el TAMANO del fichero ya copiado. Si Drive corta la copia a
    medias (pasa), antes quedaba un HDF5 truncado que `os.path.exists` daba por
    bueno para siempre y reventaba a mitad de epoca con un error de h5py que no
    dice nada. Ahora se detecta y se vuelve a copiar.
    """
    src = os.path.join(DRIVE_DATA, sesion)
    dst = os.path.join(DATA_ROOT, sesion)
    os.makedirs(dst, exist_ok=True)
    for split in ("train", "val"):
        f_src = os.path.join(src, f"data_{split}.hdf5")
        f_dst = os.path.join(dst, f"data_{split}.hdf5")
        if not os.path.exists(f_src):
            continue
        n_src = os.path.getsize(f_src)
        if os.path.exists(f_dst):
            if os.path.getsize(f_dst) == n_src:
                continue
            print(f"  [aviso] {sesion}/{split}: copia incompleta "
                  f"({os.path.getsize(f_dst)/1e6:.0f} de {n_src/1e6:.0f} MB), recopiando")
            os.remove(f_dst)
        shutil.copy(f_src, f_dst)          # data_test.hdf5 no se copia: no tiene etiquetas
        if os.path.getsize(f_dst) != n_src:
            raise IOError(f"copia fallida de {f_src}")
    return dst


def elegir_sesiones(n_sessions):
    ses = sesiones_disponibles()[:n_sessions]
    faltan = [s for s in ses
              if not os.path.exists(os.path.join(DATA_ROOT, s, "data_train.hdf5"))]
    if faltan:
        print(f"Copiando {len(faltan)} sesion(es) de Drive al disco local de la VM...")
        t0 = time.time()
        for i, s in enumerate(faltan, 1):
            copiar_sesion(s)
            print(f"  [{i}/{len(faltan)}] {s}")
        print(f"  copiadas en {time.time()-t0:.0f}s")
    return ses


_CACHE_DATOS = {}

def get_datos(cfg):
    """Devuelve (train_raw, val_raw, stats_dir, n_sesiones_reales).

    La clave de cache incluye todo lo que cambia la FORMA del tensor de entrada
    (capa_sesion anade un canal: 512 -> 513) y la longitud de text_ctc (que
    depende del objetivo), porque los shape files de ESPnet dejan de valer.

    (v6) DOS ARREGLOS IMPORTANTES:
      1. stats_dir solo distinguia "char" del resto: con objetivo "fon", "word" o
         "bpe256" reutilizaba los shape files del BPE, que tienen otras longitudes
         de text_ctc. Ahora el objetivo entra entero en el nombre.
      2. holdout_sesiones: las K ultimas sesiones se apartan de train y forman el
         conjunto de validacion. Los indices de sesion siguen siendo GLOBALES,
         asi que la capa por sesion tiene un slot para cada dia apartado (que se
         queda en la identidad porque nunca recibe gradiente).
    """
    n_sessions      = cfg["n_sessions"]
    val_igual_train = cfg["val_igual_train"]
    K               = int(cfg.get("holdout_sesiones", 0) or 0)
    clave = (n_sessions, val_igual_train, PRELOAD_RAM, cfg["capa_sesion"],
             cfg["objetivo_ctc"], cfg.get("bpe_vocab"), SOLO_SESIONES_COMPLETAS, K)
    if clave in _CACHE_DATOS:
        return _CACHE_DATOS[clave]

    sesiones = elegir_sesiones(n_sessions)
    print(f"Sesiones ({len(sesiones)}): {sesiones[:3]}{' ...' if len(sesiones) > 3 else ''}")

    if K > 0:
        con_val = [s for s in sesiones
                   if os.path.exists(os.path.join(DATA_ROOT, s, "data_val.hdf5"))]
        holdout = con_val[-K:]
        entreno = [s for s in sesiones if s not in holdout]
        print(f"  HOLDOUT de {len(holdout)} sesion(es) no vistas: {holdout}")
        print(f"  train sobre {len(entreno)} sesiones")
        idx_train = construir_indice(sesiones, "train", solo=entreno)
        idx_val   = construir_indice(sesiones, "val",   solo=holdout)
    else:
        idx_train = construir_indice(sesiones, "train")
        idx_val   = idx_train if val_igual_train else construir_indice(sesiones, "val")

    train_raw = SeeGDataset(idx_train, preload=PRELOAD_RAM)
    val_raw   = train_raw if (val_igual_train and K == 0) else SeeGDataset(idx_val, preload=PRELOAD_RAM)

    stats_dir = (f"{OUTPUT_DIR}/stats_s{len(sesiones)}"
                 f"{'_vt' if val_igual_train else ''}"
                 f"{'_ho%d' % K if K else ''}"
                 f"{'_sess' if cfg['capa_sesion'] else ''}"
                 f"_{cfg['objetivo_ctc']}"
                 f"{cfg.get('bpe_vocab', '') if cfg['objetivo_ctc'] == 'bpe256' else ''}")
    print(f"Train: {len(idx_train)} trials · Val: {len(idx_val)} trials")

    _CACHE_DATOS[clave] = (train_raw, val_raw, stats_dir, len(sesiones))
    return _CACHE_DATOS[clave]


# Comprobacion rapida: copia UNA sesion y lee un trial
_ses = sesiones_disponibles()[:1]
assert _ses, "No hay sesiones utilizables; revisa DRIVE_DATA y SOLO_SESIONES_COMPLETAS"
copiar_sesion(_ses[0])
_ds = SeeGDataset(construir_indice(_ses, "train"))
_it = _ds[0]
_x  = _it["input_features"]
print(f"\nSesion de prueba: {_ses[0]} ({len(_ds)} trials)")
print("text    :", _it["text"])
print("text_ctc:", _it["text_ctc"])
print("features:", _x.shape, _x.dtype)
# Los HDF5 de B2T'25 ya vienen z-scoreados por bloque: si esto ya esta cerca de (0, 1),
# volver a normalizar por trial (norm="trial") borra la amplitud relativa entre trials.
print(f"estadisticas del HDF5 crudo: media={_x.mean():+.3f} std={_x.std():.3f} "
      f"min={_x.min():+.2f} max={_x.max():+.2f}")

## 5. Tokenizador y modelo

Se carga OWSM una vez para quedarse con su tokenizador y el `state_dict` de las
partes no-encoder. **El encoder se inicializa al azar**, que es la conclusion
del bloque A.

Las cuatro modificaciones sobre ESPnet:

1. `ctc_weight` se inyecta en `model_conf`, no solo en el YAML, o el modelo usa
   en silencio el 0.3 por defecto de OWSM.
2. El frontend hereda de `Conv2dSubsampling`: ESPnet hace un `isinstance` en
   `e_branchformer_encoder.py:483` y llama al frontend sin la mascara de padding
   si no encaja.
3. `batch_type=numel` con `batch_bins`, porque `idim=512` genera ~96x mas
   activaciones que el frontend de audio de 80 mel.
4. `early_stopping_criterion = ["valid", "cer_ctc", "min"]`: ESPnet mide la
   paciencia sobre `valid/loss` por defecto, no sobre el criterio del mejor
   modelo.

In [ ]:
# Guarda de orden de ejecucion: esta celda depende de las secciones 3 y 4.
# Si has usado "Ejecutar celda y siguientes" desde aqui, o has saltado alguna
# celda, el error que sale sin esto es un NameError poco informativo.
for _n in ("Speech2Text", "normaliza", "SeeGDataset", "np", "gc"):
    assert _n in globals(), (
        f"Falta '{_n}'. Ejecuta las celdas de las secciones 3 y 4 antes que esta "
        f"(Entorno de ejecucion -> Ejecutar todas).")

pretrained = Speech2Text.from_pretrained(FINETUNE_MODEL, lang_sym=f"<{LANGUAGE}>", beam_size=5)
pretrain_config = vars(pretrained.s2t_train_args)
tokenizer = pretrained.tokenizer
converter = pretrained.converter
_pretrained_state_dict = {k: v.cpu().clone() for k, v in pretrained.s2t_model.state_dict().items()}
del pretrained
gc.collect()

def tokenize(text):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(text)), dtype=np.int64)

BLANK_ID = converter.token_list.index("<blank>") if "<blank>" in converter.token_list else 0

# ── Vocabulario de caracteres para el objetivo CTC alternativo ──────────────
# Solo lo que aparece en las transcripciones de B2T'25 (ingles en minusculas, sin
# puntuacion). El indice 0 se reserva para el blank de CTC.
_CHARS = list("abcdefghijklmnopqrstuvwxyz'") + [" "]
CHAR_BLANK_ID = 0
_CHAR2ID = {c: i + 1 for i, c in enumerate(_CHARS)}      # 0 = blank
_ID2CHAR = {i + 1: c for i, c in enumerate(_CHARS)}
CHAR_VOCAB = len(_CHARS) + 1                              # +1 por el blank

def tokenize_char(text):
    """Trocea en caracteres. Ignora cualquier simbolo fuera del vocabulario."""
    return np.array([_CHAR2ID[c] for c in text.lower() if c in _CHAR2ID], dtype=np.int64)

def destok_char(ids):
    """De ids de caracteres a texto (ya sin blanks; el colapso CTC se hace fuera)."""
    return "".join(_ID2CHAR[i] for i in ids if i in _ID2CHAR)


# ══════════════════════════════════════════════════════════════════════
# (v6) OBJETIVOS CTC: char · fon · bpe256 · word · bpe
# ══════════════════════════════════════════════════════════════════════
# Cada objetivo declara:
#   tok    : callable(valor_del_campo) -> np.int64[]
#   destok : callable(ids) -> str  (para el log y las metricas)
#   blank  : indice del blank en su vocabulario
#   vocab  : numero de clases (blank incluido)
#   campo  : que campo del dataset se tokeniza ("text_ctc" u "fon_ids")
#   texto  : True si destok produce texto ortografico comparable con text_raw.
#            Con "fon" es False: comparar una cadena de fonemas contra la frase
#            no significa nada, asi que la evaluacion reporta PER y deja el
#            CER/WER de texto en NaN (hace falta el decodificador con lexico).

# ── fonemas: inventario leido del propio dataset ──────────────────────
# Los ids del HDF5 se desplazan +1 para reservar el 0 como blank de CTC.
FON_NOMBRES = None      # se rellena en preparar_objetivo()
FON_VOCAB   = None

def _fon_tok(ids):
    """Fonemas del HDF5 -> ids CTC, QUITANDO EL RELLENO.

    (v12) BUG CORREGIDO, y anulaba el objetivo entero. B2T'25 guarda la
    secuencia de fonemas en un array de longitud fija rellenado con ceros
    (id 0 = BLANK del inventario). Sin quitarlo, L pasa de ~15 a ~340 y:

      1. L > T en el 70% de los trials, asi que zero_infinity=True anulaba
         su perdida EN SILENCIO. B1fon entreno con el 30% de los ejemplos.
      2. El relleno metia un grupo extra al partir por el separador '|',
         asi que el alineamiento del lexico fallaba en el 100% de las
         frases (8071 de 8072 descartadas) y no habia forma de convertir
         una hipotesis de fonemas en palabras.

    Se recortan solo los ceros FINALES, no todos: si hubiera un 0 interior
    seria un simbolo real y borrarlo cambiaria la secuencia en silencio.
    La celda 10c comprueba que no los hay.
    """
    a = np.asarray(ids, dtype=np.int64).ravel()
    nz = np.nonzero(a)[0]
    a = a[: nz[-1] + 1] if len(nz) else a[:0]
    return a + 1

def _fon_destok(ids):
    if FON_NOMBRES:
        return " ".join(FON_NOMBRES[i - 1] if 0 < i <= len(FON_NOMBRES) else f"p{i}"
                        for i in ids)
    return " ".join(str(i) for i in ids)

# ── BPE propio (sentencepiece sobre las transcripciones de train) ─────
_SP = {}          # vocab -> (modelo sentencepiece, n_clases)

def _sp_tok(v):
    sp = _SP[v][0]
    return lambda t: np.array([i + 1 for i in sp.encode(t, out_type=int)], dtype=np.int64)

def _sp_destok(v):
    sp = _SP[v][0]
    return lambda ids: sp.decode([int(i) - 1 for i in ids if int(i) > 0])

# ── palabras: vocabulario cerrado de train ────────────────────────────
WORD2ID, ID2WORD = {}, {}

def _word_tok(t):
    return np.array([WORD2ID[w] for w in t.split() if w in WORD2ID], dtype=np.int64)

def _word_destok(ids):
    return " ".join(ID2WORD[i] for i in ids if i in ID2WORD)


def objetivo_info(cfg):
    """Descriptor del objetivo CTC. Devuelve un dict (antes era una 4-tupla)."""
    o = cfg["objetivo_ctc"]
    if o == "char":
        return dict(tok=tokenize_char, destok=destok_char, blank=CHAR_BLANK_ID,
                    vocab=CHAR_VOCAB, campo="text_ctc", texto=True,
                    etiqueta=f"char({CHAR_VOCAB})")
    if o == "fon":
        assert FON_VOCAB, ("objetivo 'fon' sin preparar: ejecuta la celda de "
                           "VERIFICACIONES antes del barrido")
        # (v12) destok_texto: convierte la hipotesis de fonemas en palabras via
        # el lexico de la celda 10c, para que "fon" salga en el MISMO eje de
        # CER/WER que char/bpe/word. Sin esto solo se puede reportar un PER, que
        # no se puede comparar con nada, y el run se desperdicia. Se resuelve en
        # tiempo de llamada porque fonemas_a_texto se define despues (celda 10c).
        return dict(tok=_fon_tok, destok=_fon_destok, blank=0, vocab=FON_VOCAB,
                    campo="fon_ids", texto=False, etiqueta=f"fon({FON_VOCAB})",
                    destok_texto=globals().get("fonemas_a_texto"))
    if o == "bpe256":
        v = cfg.get("bpe_vocab", 256)
        assert v in _SP, (f"BPE de {v} clases sin entrenar: ejecuta la celda de "
                          f"VERIFICACIONES antes del barrido")
        return dict(tok=_sp_tok(v), destok=_sp_destok(v), blank=0, vocab=_SP[v][1],
                    campo="text_ctc", texto=True, etiqueta=f"bpe{v}({_SP[v][1]})")
    if o == "word":
        assert WORD2ID, ("vocabulario de palabras sin construir: ejecuta la celda "
                         "de VERIFICACIONES antes del barrido")
        return dict(tok=_word_tok, destok=_word_destok, blank=0, vocab=len(WORD2ID) + 1,
                    campo="text_ctc", texto=True, etiqueta=f"word({len(WORD2ID)+1})")
    return dict(tok=tokenize, blank=BLANK_ID, vocab=len(converter.token_list),
                destok=(lambda ids: tokenizer.tokens2text(converter.ids2tokens(list(ids)))),
                campo="text_ctc", texto=True,
                etiqueta=f"bpe({len(converter.token_list)})")


def enmascarar(x, cfg, rng):
    """Aumento tipo SpecAugment adaptado a sEEG: bandas de tiempo y de canales a cero.

    OJO: se aplica AQUI y no con el specaug de ESPnet a proposito. El specaug de
    ESPnet actua sobre el tensor de entrada completo, que con capa_sesion incluye
    el canal extra con el indice de sesion; enmascararlo lo pondria a 0 y el
    modelo creeria que todos los trials son de la sesion 0. Aqui se enmascaran
    solo los 512 canales de senal, antes de anadir el de sesion.
    """
    # ORDEN IMPORTANTE: primero ruido y offset, despues el enmascarado. Al reves,
    # el ruido rellenaba los canales que se acababan de poner a cero y el
    # enmascarado no enmascaraba nada.
    #
    # Ruido blanco: el regularizador principal del baseline de B2T'25. Con 8.000
    # frases y un encoder de 100 M, obliga a no fiarse de la amplitud exacta.
    if cfg["aug_ruido"] > 0:
        x += rng.normal(0, cfg["aug_ruido"], x.shape).astype(np.float32)
    # Offset constante por canal: un solo valor por electrodo para todo el trial.
    # Imita la deriva de linea base dentro de una sesion, que la capa por sesion
    # no puede corregir porque cambia de un trial a otro.
    if cfg.get("aug_offset", 0) > 0:
        x += rng.normal(0, cfg["aug_offset"], (1, x.shape[1])).astype(np.float32)

    T = x.shape[0]
    for _ in range(cfg["aug_n_tiempo"]):                 # bandas temporales
        w = rng.integers(0, max(1, int(cfg["aug_max_tiempo"] * T)) + 1)
        if w > 0 and T > w:
            t0 = rng.integers(0, T - w)
            x[t0:t0 + w, :] = 0.0
    # Electrodos: subconjunto ALEATORIO, no una banda contigua. El orden de los
    # electrodos es arbitrario (por eso tampoco se convoluciona sobre ese eje),
    # asi que borrar c0:c0+w no tiene el sentido fisico que si tiene en audio,
    # donde la banda de frecuencias contigua es una unidad real.
    # OJO (v6): aqui NO se sortea el ancho. n_canales x max_canales es el numero
    # EXACTO de electrodos que se anulan en cada trial (con los valores de v3/v5,
    # 2 x 20 = 40 de 512, siempre). El nombre "max" viene de SpecAugment y
    # despista: en la memoria hay que describirlo como "40 electrodos elegidos al
    # azar por trial", no como "hasta 40".
    n_c = int(cfg["aug_n_canales"] * cfg["aug_max_canales"])
    if n_c > 0:
        x[:, rng.choice(512, size=min(n_c, 512), replace=False)] = 0.0
    return x


def preparar_speech(d, cfg, aug=False, rng=None):
    """Tensor de entrada al modelo: (T, 512) o (T, 513) si hay capa por sesion.

    El indice de sesion viaja como un canal extra constante. Es la forma menos invasiva
    de meterlo: el pipeline de ESPnet solo transporta 'speech', y el padding va al final
    (con ceros), asi que el frame 0 siempre es dato real y de ahi lo lee el frontend.
    """
    x = normaliza(d["input_features"], cfg["norm"])
    if aug:
        x = enmascarar(x, cfg, rng)
    # Suavizado gaussiano temporal. OJO: esto NO es aumento, es preprocesado, asi
    # que va tambien en validacion e inferencia. Va DESPUES del ruido a proposito:
    # asi el suavizado tiene algo que suavizar, que es como lo aplica B2T'25.
    if cfg.get("suavizado", 0) > 0:
        from scipy.ndimage import gaussian_filter1d
        x = gaussian_filter1d(x, cfg["suavizado"], axis=0, mode="nearest").astype(np.float32)
    if cfg["capa_sesion"]:
        sid = np.full((x.shape[0], 1), float(d["session_idx"]), dtype=np.float32)
        x = np.concatenate([x, sid], axis=1)
    return x


def make_data_info(cfg, aug=False):
    # text y text_prev SIEMPRE en BPE: alimentan el decoder de atencion de OWSM.
    # text_ctc usa el tokenizador del objetivo elegido.
    obj = objetivo_info(cfg)
    tok_ctc, campo = obj["tok"], obj["campo"]
    aug = aug and cfg["aug"]
    rng = np.random.default_rng(cfg["seed"]) if aug else None
    return {
        "speech":    lambda d: preparar_speech(d, cfg, aug, rng),
        "text":      lambda d: tokenize(d["text"]),
        "text_prev": lambda d: tokenize(d["text_prev"]),
        "text_ctc":  lambda d: tok_ctc(d[campo]),
    }

print("Tokenizador y config base cargados")
print("  ctc_weight original de OWSM:", pretrain_config.get("model_conf", {}).get("ctc_weight"))
print(f"  BPE: {len(converter.token_list)} clases (blank id {BLANK_ID}) · "
      f"caracteres: {CHAR_VOCAB} clases (blank id {CHAR_BLANK_ID})")
print("  fon / bpe256 / word se preparan en la celda de VERIFICACIONES")

In [ ]:
def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def _subsample_mask(x_mask, subsample, target_len):
    """Recorta/rellena la mascara para que case EXACTAMENTE con la longitud de x tras el submuestreo."""
    if x_mask is None:
        return None
    m = x_mask[:, :, ::subsample]
    if m.size(2) > target_len:
        m = m[:, :, :target_len]
    elif m.size(2) < target_len:
        m = torch.nn.functional.pad(m, (0, target_len - m.size(2)), value=False)
    return m


class CapaSesion(torch.nn.Module):
    """Transformada afin por sesion + softsign (capa 'day-specific' del baseline de B2T'25).

    Por que hace falta: los 45 dias de grabacion abarcan 20 meses y la senal que produce
    un mismo fonema cambia de un dia a otro (deriva de los electrodos, micromovimientos
    del array). Un unico frontend compartido tiene que aprender N mapeos senal->fonema
    contradictorios a la vez y ademas adivinar de que dia es cada trial. Con una matriz
    por sesion, cada dia se alinea a un espacio comun y el resto del modelo ve un unico
    problema coherente.

    Se inicializa a la identidad, asi que al empezar solo aplica el softsign.
    Coste: n_sesiones x 512 x 512 parametros (~2.6 M con 10 sesiones, ~12 M con 45).
    """

    def __init__(self, n_sesiones, dim=512):
        super().__init__()
        self.n_sesiones = n_sesiones
        self.W = torch.nn.Parameter(torch.eye(dim).unsqueeze(0).repeat(n_sesiones, 1, 1))
        self.b = torch.nn.Parameter(torch.zeros(n_sesiones, dim))

    def forward(self, x):
        # x: (B, T, dim+1). El ultimo canal es el indice de sesion, constante en el tiempo.
        sid = x[:, 0, -1].round().long().clamp_(0, self.n_sesiones - 1)
        x = x[:, :, :-1]
        # baddbmm: b[sid] + x @ W[sid], por muestra del batch
        x = torch.baddbmm(self.b[sid].unsqueeze(1), x, self.W[sid])
        return torch.nn.functional.softsign(x)


def build_frontend(kind, pos_enc, capa_sesion=None, cfg=None):
    """Frontend (encoder.embed) que reemplaza al Conv2dSubsampling original de mel.

    IMPORTANTE: heredan de Conv2dSubsampling (aunque no usen su __init__) porque el
    encoder de ESPnet decide si pasar la mascara al embed mirando
    `isinstance(self.embed, Conv2dSubsampling)`. Si no se hereda de ahi, ESPnet llama
    a embed(x) SIN mascara (asumiendo que es un embedding de texto que no submuestrea)
    y revienta con "forward() missing 1 required positional argument: 'x_mask'".
    """
    import torch.nn as nn

    if kind == "conv2d":
        class Conv2dFrontend(Conv2dSubsampling):
            # OJO: ~66 GFLOP/muestra, el 85-90% del coste del modelo.
            def __init__(self):
                super().__init__(512, 384, dropout_rate=0.0, pos_enc=pos_enc)
                self.sesion = capa_sesion
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                return super().forward(x, x_mask)
        return Conv2dFrontend()

    elif kind == "linear":
        class LinearFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384, subsample=4):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.proj = nn.Linear(in_dim, out_dim)
                self.subsample = subsample
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.proj(x)
                x = x[:, ::self.subsample, :]
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
                return x, x_mask
        return LinearFrontend()

    elif kind == "conv1d":
        class Conv1dFrontend(Conv2dSubsampling):
            def __init__(self, in_dim=512, out_dim=384):
                torch.nn.Module.__init__(self)   # NO llamar a Conv2dSubsampling.__init__
                self.sesion = capa_sesion
                self.conv = nn.Sequential(
                    nn.Conv1d(in_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                    nn.Conv1d(out_dim, out_dim, kernel_size=3, stride=2, padding=1), nn.ReLU(),
                )
                self.pos_enc = pos_enc
            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.conv(x.transpose(1, 2)).transpose(1, 2)
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, 4, x.size(1))
                return x, x_mask
        return Conv1dFrontend()

    elif kind == "deep":
        # Frontend convolucional profundo, pensado para sEEG (no para audio).
        #   1. proyeccion ESPACIAL: Linear(512 -> d) que mezcla los 256 electrodos.
        #      A diferencia de conv2d, que convoluciona a lo largo del eje de
        #      electrodos como si fuera un eje de frecuencias, aqui la mezcla es
        #      todos-con-todos, que es lo correcto cuando el orden de los
        #      electrodos es arbitrario.
        #   2. bloques TEMPORALES Conv1d con LayerNorm + GELU y conexiones
        #      residuales. Las convoluciones ven todos los frames (nada de
        #      decimacion), y el submuestreo se hace con stride.
        #   3. proyeccion final a 384 + codificacion posicional de OWSM.
        d        = (cfg or {}).get("deep_dim", 512)
        n_res    = (cfg or {}).get("deep_bloques", 2)   # bloques residuales por etapa
        kernel   = (cfg or {}).get("deep_kernel", 5)
        dropout  = (cfg or {}).get("deep_dropout", 0.1)
        subsample = (cfg or {}).get("subsample", 4)     # 4 = dos etapas de stride 2

        class BloqueTemporal(nn.Module):
            def __init__(self, dim, k, stride):
                super().__init__()
                self.conv = nn.Conv1d(dim, dim, k, stride=stride, padding=k // 2)
                self.norm = nn.LayerNorm(dim)
                self.act = nn.GELU()
                self.drop = nn.Dropout(dropout)
                self.residual = (stride == 1)
            def forward(self, x):                        # x: (B, T, dim)
                y = self.conv(x.transpose(1, 2)).transpose(1, 2)
                y = self.drop(self.act(self.norm(y)))
                return x + y if self.residual else y

        class FrontendProfundo(Conv2dSubsampling):
            def __init__(self):
                torch.nn.Module.__init__(self)
                self.sesion = capa_sesion
                self.subsample = subsample
                self.espacial = nn.Sequential(
                    nn.Linear(512, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout))
                bloques = []
                n_etapas = {1: 0, 2: 1, 4: 2, 8: 3}[subsample]
                for _ in range(n_etapas):
                    bloques.append(BloqueTemporal(d, kernel, stride=2))
                    bloques += [BloqueTemporal(d, kernel, 1) for _ in range(n_res)]
                if n_etapas == 0:
                    bloques += [BloqueTemporal(d, kernel, 1) for _ in range(n_res)]
                self.bloques = nn.ModuleList(bloques)
                self.salida = nn.Linear(d, 384)
                self.pos_enc = pos_enc

            def forward(self, x, x_mask):
                if self.sesion is not None:
                    x = self.sesion(x)
                x = self.espacial(x)
                # Anular el padding: tras la capa espacial (con sesgo) los frames
                # de relleno dejan de ser cero y se colarian en las convoluciones.
                if x_mask is not None:
                    x = x * x_mask.transpose(1, 2).to(x.dtype)
                for b in self.bloques:
                    x = b(x)
                x = self.salida(x)
                x = self.pos_enc(x)
                x_mask = _subsample_mask(x_mask, self.subsample, x.size(1))
                return x, x_mask

        return FrontendProfundo()

    raise ValueError(f"Frontend desconocido: {kind}")


def make_build_model_fn(cfg, n_sesiones=1, verbose=True):
    """Devuelve el build_model_fn que ESPnet-EZ usara para ESTE experimento."""
    def build_model_fn(args):
        from espnet2.tasks.s2t import S2TTask

        # ── ARREGLO: el ctc_weight del panel tiene que llegar al modelo ──
        conf = dict(pretrain_config)
        conf["model_conf"] = {**conf.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

        # ── InterCTC: perdidas CTC auxiliares en capas intermedias del encoder ──
        # Da senal de gradiente directa a las capas de abajo en vez de que tenga
        # que propagarse desde el final. Es la receta estandar contra el
        # subajuste en encoders profundos entrenados con CTC.
        if cfg["interctc_weight"] > 0:
            conf["encoder_conf"] = {**conf.get("encoder_conf", {}),
                                    "interctc_layer_idx": list(cfg["interctc_layers"])}
            conf["model_conf"]["interctc_weight"] = cfg["interctc_weight"]

        # ── (v7) encoder recortado ────────────────────────────────────
        # El encoder de owsm_v3.1_ebf_base tiene 6 bloques E-Branchformer.
        # Con init aleatorio se puede pedir menos: los pesos no se cargan, asi
        # que nada tiene que encajar. Mide si hace falta toda la arquitectura o
        # solo una parte, que es la pregunta natural despues del bloque A.
        _ncap = cfg.get("n_capas_encoder")
        if _ncap is not None:
            assert cfg.get("init_encoder") == "random", (
                "n_capas_encoder solo vale con init_encoder='random': con los "
                "pesos de OWSM el state_dict no encajaria")
            conf["encoder_conf"] = {**conf.get("encoder_conf", {}),
                                    "num_blocks": int(_ncap)}
            if cfg["interctc_weight"] > 0:
                # interctc_layer_idx valido: 1..N-1
                val = [l for l in cfg["interctc_layers"] if 1 <= l <= int(_ncap) - 1]
                conf["encoder_conf"]["interctc_layer_idx"] = val or [max(1, int(_ncap) - 1)]
            print(f"  [v7] encoder recortado a {_ncap} bloques "
                  f"(interctc en {conf['encoder_conf']['interctc_layer_idx']})")

        model = S2TTask.build_model(argparse.Namespace(**conf))

        # ── (v6) init_encoder: el control cientifico ──────────────────
        # Con "random" NO se cargan los pesos de OWSM. Todo lo demas es
        # identico. Si este punto empata con el preentrenado, el TFG concluye
        # que OWSM no transfiere, y esa es una conclusion tan valida como la
        # contraria (pero hay que medirla, no suponerla).
        if cfg.get("init_encoder", "owsm") == "owsm":
            model.load_state_dict(_pretrained_state_dict, strict=False)
        else:
            print("  [control] encoder con inicializacion ALEATORIA (sin pesos de OWSM)")

        # sEEG entra directa: fuera frontend mel y normalizacion global
        model.frontend = None
        model.normalize = None

        pos_enc = model.encoder.embed.out[1]
        sesion = CapaSesion(n_sesiones) if cfg["capa_sesion"] else None
        model.encoder.embed = build_frontend(cfg["frontend"], pos_enc, sesion, cfg)

        # ── (v6) cabeza CTC del objetivo elegido + error_calculator coherente ──
        # Vale para char, fon, bpe256 y word, no solo para char.
        obj = objetivo_info(cfg)
        if cfg["objetivo_ctc"] != "bpe" and model.ctc is not None:
            from espnet2.asr.ctc import CTC
            from espnet.nets.pytorch_backend.transformer.subsampling import Conv2dSubsampling  # noqa
            from espnet.nets.e2e_asr_common import ErrorCalculator
            enc_dim = model.ctc.ctc_lo.in_features
            # ctc_lo pasa de (enc_dim -> 50002) a (enc_dim -> vocab del objetivo).
            # Se entrena desde cero: es la capa que hace que BPE-50k sea inviable
            # con 8.000 frases y char/fon si lo sean.
            model.ctc = CTC(odim=obj["vocab"], encoder_output_size=enc_dim,
                            zero_infinity=True)
            # El cer_ctc del train.log lo calcula el error_calculator con SU
            # token_list; si no se le da la del objetivo, las cifras del log no
            # quieren decir nada.
            if cfg["objetivo_ctc"] == "char":
                lista = ["<blank>"] + _CHARS
            elif cfg["objetivo_ctc"] == "fon":
                lista = ["<blank>"] + (FON_NOMBRES or
                                       [f"p{i}" for i in range(1, obj["vocab"])])
            elif cfg["objetivo_ctc"] == "word":
                lista = ["<blank>"] + [ID2WORD[i] for i in range(1, len(ID2WORD) + 1)]
            else:
                sp = _SP[cfg.get("bpe_vocab", 256)][0]
                lista = ["<blank>"] + [sp.id_to_piece(i) for i in range(sp.get_piece_size())]
            model.error_calculator = ErrorCalculator(
                lista, " ", "<blank>", report_cer=True, report_wer=True)
            model.token_list = lista

        model.train()
        freeze_all(model)
        # LoRA solo tiene sentido con la base CONGELADA: aprende un delta de rango bajo
        # sobre pesos fijos. Si el encoder tambien se entrena, W y BA se solapan y solo
        # se anaden parametros y ciclos de merge/unmerge sin ganar nada.
        _entero = (cfg["unfreeze_encoder"] and cfg.get("unfreeze_ultimas") is None
                   and cfg.get("unfreeze_n_layers") is None)
        if cfg["usar_lora"] and _entero:
            if verbose:
                print("  [nota] usar_lora=True con el encoder entero descongelado no aporta: "
                      "se desactiva LoRA")
        elif cfg["usar_lora"]:
            create_lora_adapter(model, target_modules=LORA_TARGET)

        for p in model.encoder.embed.parameters():   # frontend nuevo (incl. capa de sesion)
            p.requires_grad = True
        if model.ctc is not None:                    # ESPnet lo pone a None si ctc_weight==0
            for p in model.ctc.parameters():
                p.requires_grad = True

        # ── (v6) descongelacion parcial: por ARRIBA, no por abajo ─────
        # BUG CORREGIDO: `encoders[:N]` descongelaba las N PRIMERAS capas, las
        # mas cercanas a la entrada. En fine-tuning se descongelan las de
        # ARRIBA, las mas cercanas a la salida y las mas especificas de la
        # tarea. Con encoders[:2] se estaba haciendo justo lo contrario.
        n_ult   = cfg.get("unfreeze_ultimas")
        n_prim  = cfg.get("unfreeze_n_layers")     # obsoleto
        if cfg["unfreeze_encoder"]:
            capas = model.encoder.encoders
            if n_ult is None and n_prim is None:
                for p in model.encoder.parameters():
                    p.requires_grad = True
                sel = f"encoder entero ({len(capas)} capas)"
            elif n_ult is not None:
                for layer in capas[-int(n_ult):]:
                    for p in layer.parameters():
                        p.requires_grad = True
                # after_norm es la LayerNorm final del encoder: va con las de arriba
                if getattr(model.encoder, "after_norm", None) is not None:
                    for p in model.encoder.after_norm.parameters():
                        p.requires_grad = True
                sel = f"ultimas {int(n_ult)} de {len(capas)} capas"
            else:
                for layer in capas[:int(n_prim)]:
                    for p in layer.parameters():
                        p.requires_grad = True
                sel = f"[OBSOLETO] primeras {int(n_prim)} de {len(capas)} capas"
            if verbose:
                print(f"  encoder: {sel}")

        if verbose:
            total, trainable = count_trainable(model)
            extra = f" + capa_sesion({n_sesiones})" if cfg["capa_sesion"] else ""
            print(f"  init={cfg.get('init_encoder', 'owsm')}")
            print(f"  frontend={cfg['frontend']}{extra} · objetivo_ctc={obj['etiqueta']} · "
                  f"lora={cfg['usar_lora']} · "
                  f"encoder={'descongelado' if cfg['unfreeze_encoder'] else 'congelado'} · "
                  f"ctc_weight={cfg['ctc_weight']} · norm={cfg['norm']}")
            print(f"  {trainable:,} entrenables / {total:,} totales ({trainable/total*100:.2f}%)")
            if model.ctc is None:
                print("  [nota] ctc_weight=0 -> ESPnet desactiva la cabeza CTC")
            if getattr(model, "decoder", None) is None:
                print("  [nota] ctc_weight=1 -> ESPnet desactiva el decoder "
                      "(solo CTC; la evaluacion sera greedy)")
        return model

    return build_model_fn


# ── LR diferencial: grupos de parametros en el optimizador ────────────
def set_encoder_lr_scale(scale):
    """scale<1.0 -> el encoder preentrenado entrena con lr*scale; el frontend nuevo, CTC y LoRA con lr."""
    from espnet2.tasks.s2t import S2TTask

    if scale is None or scale == 1.0:
        if "build_optimizers" in S2TTask.__dict__:
            del S2TTask.build_optimizers      # restaura la implementacion original
        return

    OPTIMS = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW, "sgd": torch.optim.SGD}

    def build_optimizers(cls, args, model):
        if args.optim not in OPTIMS:
            raise ValueError(f"enc_lr_scale no soportado con optim={args.optim}")
        base_lr = args.optim_conf.get("lr", 1e-3)
        conf = {k: v for k, v in args.optim_conf.items() if k != "lr"}
        nuevos, preentrenados = [], []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            es_nuevo = (name.startswith("encoder.embed") or name.startswith("ctc.")
                        or "lora_" in name)
            (nuevos if es_nuevo else preentrenados).append(p)
        grupos = [{"params": nuevos, "lr": base_lr},
                  {"params": preentrenados, "lr": base_lr * scale}]
        print(f"  LR diferencial: {len(nuevos)} tensores a {base_lr:g} · "
              f"{len(preentrenados)} a {base_lr*scale:g}")
        return [OPTIMS[args.optim](grupos, lr=base_lr, **conf)]

    S2TTask.build_optimizers = classmethod(build_optimizers)

print("Constructores de modelo listos.")

In [ ]:
YAML_BASE = """
use_lora: true

rir_scp: null
noise_scp: null
speech_volume_normalize: null
non_linguistic_symbols: null

preprocessor_conf:
  speech_name: speech
  text_name: text

seed: 2024
num_workers: 0
ngpu: 1
batch_type: sorted
batch_size: 8
accum_grad: 1
max_epoch: 1
patience: null
init: null
best_model_criterion:
- [valid, loss, min]
keep_nbest_models: 1
use_amp: true

optim: adam
optim_conf:
    lr: 0.001
    weight_decay: 0.000001
scheduler: warmuplr
scheduler_conf:
    warmup_steps: 20

specaug: null
ctc_weight: 0.0
grad_clip: 5.0
"""

with open("/content/finetune_b2t25.yaml", "w") as f:
    f.write(YAML_BASE)


def build_finetune_config(cfg, stats_dir):
    ft = ez.config.update_finetune_config(
        "s2t", copy.deepcopy(pretrain_config), "/content/finetune_b2t25.yaml")

    # ── entorno ──
    ft["ngpu"] = NGPU
    ft["use_amp"] = USE_AMP
    ft["num_workers"] = cfg["num_workers"]
    # cudnn_benchmark=True es CONTRAPRODUCENTE aqui: con batch_type sorted cada batch
    # tiene una longitud distinta, y cudnn vuelve a buscar el mejor algoritmo para cada
    # forma nueva (143 busquedas exhaustivas en la primera epoca).
    ft["cudnn_benchmark"] = False
    ft["cudnn_deterministic"] = False
    ft["multiple_iterator"] = False
    ft["iterator_type"] = "sequence"
    ft["log_interval"] = 10
    ft["num_iters_per_epoch"] = None
    ft["seed"] = cfg["seed"]
    ft["resume"] = True
    ft["keep_nbest_models"] = 1
    # OJO: en ESPnet, patience NO se mide sobre best_model_criterion. Usa un
    # parametro aparte, early_stopping_criterion, cuyo defecto es
    # ("valid", "loss", "min"). Con sobreajuste el loss de validacion sube
    # mientras el CER sigue bajando, asi que el defecto corta el entrenamiento
    # antes de tiempo: v3 paro en la epoca 38 de 60 con el mejor CER en la 37 y
    # una pendiente de -0.0035 por epoca, o sea aun mejorando.
    #
    # patience_start_epoch evita que se cuente durante el warmup ruidoso (epocas
    # 1-11 en v3 tuvieron delta erratica). Esperar hasta la 15 es seguro.
    # (v6) patience_start_epoch se lee del cfg. Antes estaba fijo a 15 aqui y
    # se ignoraba lo que pusieras en el barrido.
    ft["patience"] = cfg.get("patience")
    ft["patience_start_epoch"] = cfg.get("patience_start_epoch", 15)
    ft["early_stopping_criterion"] = ["valid", "cer_ctc", "min"]

    # ── criterio para elegir el mejor checkpoint ──
    # ARREGLO: con ctc_weight bajo, 'valid loss' es sobre todo perdida de atencion y
    # toca fondo hacia la epoca 4, cuando el cer_ctc todavia esta bajando. Seleccionar
    # por cer_ctc guarda el modelo que de verdad es mejor en la tarea.
    if cfg["criterio"] == "cer_ctc" and cfg["ctc_weight"] > 0:
        ft["best_model_criterion"] = [["valid", "cer_ctc", "min"]]
    else:
        ft["best_model_criterion"] = [["valid", "loss", "min"]]

    # ── batching ──
    ft["batch_type"] = cfg["batch_type"]
    if cfg["batch_type"] == "numel":
        ft["batch_bins"] = cfg["batch_bins"]
    else:
        ft["batch_size"] = cfg["batch_size"]
    ft["accum_grad"] = cfg["accum_grad"]

    # ── optimizacion ──
    ft["max_epoch"] = cfg["max_epoch"]
    ft["scheduler_conf"]["warmup_steps"] = cfg["warmup"]
    ft["optim"] = cfg["optim"]
    ft["optim_conf"]["lr"] = cfg["lr"]
    ft["optim_conf"]["weight_decay"] = cfg["weight_decay"]
    ft["grad_clip"] = cfg["grad_clip"]
    ft["ctc_weight"] = cfg["ctc_weight"]          # informativo, el que manda es model_conf
    ft["model_conf"] = {**ft.get("model_conf", {}), "ctc_weight": cfg["ctc_weight"]}

    # ── shape files ──
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    ft["train_shape_file"] = [f"{stats_dir}/train/{n}" for n in nombres]
    ft["valid_shape_file"] = [f"{stats_dir}/valid/{n}" for n in nombres]
    return ft


def shape_files(stats_dir):
    nombres = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
    return ([f"{stats_dir}/train/{n}" for n in nombres]
            + [f"{stats_dir}/valid/{n}" for n in nombres])

print("Constructor de config listo.")

In [ ]:
class CapturaLog:
    """Redirige el logging de ESPnet a un fichero; en pantalla, solo el resumen por epoca."""

    CLAVES = ("batch:", "epoch results", "epoch started", "Saving", "best",
              "There are no improvements", "Stop training", "The training was finished")

    def __init__(self, path):
        self.path = path
        self.previos = None

    def __enter__(self):
        root = logging.getLogger()
        self.previos = (root.handlers[:], root.level)
        root.handlers = []
        root.setLevel(logging.INFO)

        fh = logging.FileHandler(self.path, mode="a", encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
        root.addHandler(fh)

        claves = self.CLAVES
        class Resumen(logging.Filter):
            def filter(self, record):
                return any(c in record.getMessage() for c in claves)
        sh = logging.StreamHandler()
        sh.addFilter(Resumen())
        sh.setFormatter(logging.Formatter("    %(message)s"))
        root.addHandler(sh)
        return self

    def __exit__(self, *exc):
        root = logging.getLogger()
        for h in root.handlers:
            try: h.close()
            except Exception: pass
        root.handlers, root.level = self.previos
        return False


PAR = re.compile(r"([a-zA-Z_][a-zA-Z_0-9]*)=(-?[\d.]+(?:[eE][-+]?\d+)?)")

def parse_train_log(path):
    """Extrae una fila por epoca del train.log de ESPnet."""
    filas = []
    if not os.path.exists(path):
        return pd.DataFrame()
    with open(path, encoding="utf-8", errors="ignore") as f:
        for linea in f:
            m = re.search(r"(\d+)epoch results:(.*)", linea)
            if not m:
                continue
            epoca, resto = int(m.group(1)), m.group(2)
            fila = {"epoch": epoca}
            partes = re.split(r"\[valid\]", resto)
            for prefijo, trozo in zip(["train_", "valid_"], partes):
                trozo = trozo.replace("[train]", "")
                for k, v in PAR.findall(trozo):
                    if k in ("time", "total_count"):
                        continue
                    fila[prefijo + k] = float(v)
            filas.append(fila)
    df = pd.DataFrame(filas)
    if len(df):
        df = df.drop_duplicates(subset="epoch", keep="last").sort_values("epoch")
    return df

print("Utilidades de log listas.")

In [ ]:
_ULTIMO_SE_GREEDY = float("nan")   # (v6) para guardar en el CSV
def _norm(t):
    t = t.lower().replace("\x00", "")
    t = re.sub(r"<[^>]+>", " ", t)
    t = t.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", t).strip()

def _lev(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            old = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
            prev = old
    return dp[-1]

def cer(ref, hyp):
    r, h = _norm(ref), _norm(hyp)
    return 0.0 if len(r) == 0 else _lev(list(r), list(h)) / len(r)

def wer(ref, hyp):
    r, h = _norm(ref).split(), _norm(hyp).split()
    return 0.0 if len(r) == 0 else _lev(r, h) / len(r)


def find_best_checkpoint(exp_dir):
    for name in ["valid.cer_ctc.best.pth", "valid.loss.best.pth",
                 "valid.acc.best.pth", "train.loss.best.pth"]:
        p = os.path.join(exp_dir, name)
        if os.path.exists(p):
            return p
    cands = sorted(glob.glob(os.path.join(exp_dir, "*epoch.pth")), key=os.path.getmtime)
    if not cands:
        raise FileNotFoundError(f"No hay checkpoints en {exp_dir}")
    return cands[-1]


def promediar_capa_sesion(modelo, sesiones_vistas):
    """(v6) Sustituye la matriz de las sesiones NO vistas por la media de las vistas.

    Es una de las dos formas razonables de evaluar un dia nuevo: la otra es
    dejar su slot en la identidad (que es lo que pasa solo, porque nunca recibe
    gradiente). Compararlas es parte del experimento de generalizacion.
    """
    capa = getattr(modelo.encoder.embed, "sesion", None)
    if capa is None:
        return modelo
    with torch.no_grad():
        idx = torch.tensor(sorted(sesiones_vistas), device=capa.W.device)
        W_m, b_m = capa.W[idx].mean(0), capa.b[idx].mean(0)
        for s in range(capa.n_sesiones):
            if s not in sesiones_vistas:
                capa.W[s].copy_(W_m); capa.b[s].copy_(b_m)
    print(f"    [holdout] capa de sesion de los dias no vistos = media de {len(idx)} dias")
    return modelo


def cargar_modelo_entrenado(cfg, exp_dir, n_sesiones):
    ckpt = find_best_checkpoint(exp_dir)
    modelo = make_build_model_fn(cfg, n_sesiones, verbose=False)(None).to(DEVICE)
    try:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=True)
    except Exception:
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    faltan = modelo.load_state_dict(state, strict=False)
    if faltan.missing_keys:
        print(f"    [aviso] {len(faltan.missing_keys)} tensores sin cargar del checkpoint, "
              f"p.ej. {faltan.missing_keys[0]}")
    return modelo.eval()


@torch.no_grad()
def evaluar_ctc_greedy(cfg, modelo, val_raw, n=EVAL_N_GREEDY, mostrar=3):
    """Decodificacion CTC greedy: argmax por frame, colapsar repeticiones, quitar blanks.
    No pasa por el decoder, asi que mide SOLO lo que el encoder extrae del sEEG.

    Devuelve DOS cifras, porque no son la misma:
      - cer_ctc_micro: exactamente como lo calcula ESPnet en el train.log (cadena de
        tokens BPE concatenados, sum(errores)/sum(longitudes)). Es la unica comparable
        con la columna cer_ctc de las curvas.
      - cer_texto: sobre el texto detokenizado y normalizado, media por trial. Es la
        que se parece a un CER "de verdad", pero la hunden los trials cortos que fallan.
    """
    from itertools import groupby
    obj = objetivo_info(cfg)
    tok_ctc, blank, destok, campo = obj["tok"], obj["blank"], obj["destok"], obj["campo"]
    dtxt = obj.get("destok_texto")      # (v12) solo lo trae "fon"
    if dtxt:
        print(f"    [{obj['etiqueta']}] hipotesis convertidas a palabras con el "
              f"lexico ({len(globals().get("LEXICO", {}))} entradas): CER/WER comparables con char")
    n = len(val_raw) if n is None else min(n, len(val_raw))
    idxs = np.linspace(0, len(val_raw) - 1, n).astype(int)   # repartidos, no los n primeros

    err_bpe = ref_bpe = 0
    cers_txt, wers_txt, ejemplos = [], [], []
    for i in idxs:
        d = val_raw[int(i)]
        x = preparar_speech(d, cfg)
        speech = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        lens = torch.tensor([speech.shape[1]], dtype=torch.long, device=DEVICE)
        enc, _ = modelo.encode(speech, lens)
        if isinstance(enc, tuple):
            enc = enc[0]
        ids = modelo.ctc.ctc_lo(enc).argmax(-1)[0].tolist()
        hyp_ids = [k for k, _ in groupby(ids) if k != blank]   # colapso CTC estandar
        ref_ids = tok_ctc(d[campo]).tolist()

        # cer_ctc(micro): identico a como lo calcula ESPnet, sobre la unidad del objetivo
        h = destok(hyp_ids)
        r = destok(ref_ids)
        err_bpe += _lev(list(h), list(r)); ref_bpe += len(r)

        hip = destok(hyp_ids)
        # (v12) con objetivo "fon" la hipotesis es una cadena de fonemas y
        # compararla con la frase no significa nada. Si el lexico de la celda 10c
        # esta construido, destok_texto la pasa a palabras y entonces SI se puede
        # medir CER/WER en el mismo eje que el resto. Si no lo esta, se reporta
        # solo el PER, como antes.
        hip_txt = dtxt(hyp_ids) if dtxt else hip
        if obj["texto"] or dtxt:
            cers_txt.append(cer(d["text_raw"], hip_txt))
            wers_txt.append(wer(d["text_raw"], hip_txt))
        if len(ejemplos) < mostrar:
            ejemplos.append((_norm(d["text_raw"]) if (obj["texto"] or dtxt)
                             else destok(ref_ids), hip_txt))

    for ref, hip in ejemplos:
        print(f"    [ctc] REF: {ref}\n    [ctc] HYP: {hip}")
    if not obj["texto"]:
        print(f"    (objetivo {obj['etiqueta']}: la cifra micro es un PER, "
              f"no un CER de texto)")
        if not dtxt:
            return err_bpe / max(1, ref_bpe), float("nan"), float("nan")
    c = np.array(cers_txt)
    # (v6) error estandar e IC95 de la media, para saber si una diferencia entre
    # dos experimentos es real o es ruido de muestreo sobre n trials.
    se = c.std(ddof=1) / np.sqrt(len(c)) if len(c) > 1 else float("nan")
    print(f"    por trial: mediana={np.median(c):.3f} p10={np.percentile(c,10):.3f} "
          f"p90={np.percentile(c,90):.3f} · {(c<0.2).mean()*100:.0f}% por debajo de 0.2")
    print(f"    n={len(c)} · SE={se:.4f} · IC95=±{1.96*se:.4f}  "
          f"(dos runs con CERs que difieran menos que la suma de sus IC95 "
          f"no se pueden distinguir con esta n)")
    global _ULTIMO_SE_GREEDY
    _ULTIMO_SE_GREEDY = se
    return err_bpe / max(1, ref_bpe), float(c.mean()), float(np.mean(wers_txt))

In [ ]:
def asegurar_stats(trainer, stats_dir):
    faltan = [p for p in shape_files(stats_dir)
              if not os.path.exists(p) or os.path.getsize(p) == 0]
    if not faltan:
        print("  shape files ya existen, se salta collect_stats")
        return
    print("  recopilando estadisticas (solo la primera vez por configuracion de datos)...")
    nivel = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)
    try:
        trainer.collect_stats()
    finally:
        logging.getLogger().setLevel(nivel)
    print("  collect_stats OK")

## 6. Preparacion del objetivo

Construye el vocabulario cerrado de palabras de train y mide la tasa de **OOV**
en validacion, que es el suelo duro del WER de este modelo: son palabras que no
puede escribir por construccion. Es un techo estructural, no un problema de
entrenamiento, y hay que reportarlo junto al WER.

In [ ]:
_tr, _va, _, _ = get_datos(CFG)
_textos = [_norm(t) for t in _tr.textos()]
print(f"{len(_textos)} frases de entrenamiento · {len(_va)} trials de validacion\n")

import collections
_pal = collections.Counter(w for t in _textos for w in t.split())
WORD2ID = {w: i + 1 for i, w in enumerate(sorted(_pal))}     # 0 = blank
ID2WORD = {i: w for w, i in WORD2ID.items()}
_hapax = sum(1 for w, c in _pal.items() if c == 1)
print(f"[word] {len(WORD2ID)} palabras distintas · {sum(_pal.values())} tokens · "
      f"{_hapax} aparecen UNA sola vez ({_hapax/len(_pal)*100:.0f}%)")

# ── OOV de validacion = suelo del WER ─────────────────────────────────
_tok = _oov = _frases = 0
for i in range(len(_va)):
    ws = _norm(_va[i]["text_raw"]).split()
    n = sum(1 for w in ws if w not in WORD2ID)
    _tok += len(ws); _oov += n; _frases += (n > 0)
SUELO_WER = _oov / _tok
print(f"\n[OOV] {_oov}/{_tok} tokens ({SUELO_WER*100:.1f}%) · "
      f"{_frases}/{len(_va)} frases afectadas ({_frases/len(_va)*100:.0f}%)")
print(f"      -> este modelo NO PUEDE bajar de {SUELO_WER*100:.1f}% de WER.")

# OJO: _word_tok DESCARTA las palabras OOV del objetivo, asi que el modelo nunca
# se entrena para producirlas; pero la referencia de las metricas es text_raw
# COMPLETO, asi que si se le penalizan. La comparacion con char es justa.

_r, _malos = [], 0
for i in range(0, len(_tr), max(1, len(_tr) // 400)):
    d = _tr[i]
    T = int(np.ceil(d["input_features"].shape[0] / CFG["subsample"]))
    L = len(_word_tok(d["text_ctc"]))
    _r.append(T / max(1, L)); _malos += (L > T)
print(f"\n[CTC] T/L medio={np.mean(_r):.1f} · minimo={min(_r):.1f} · "
      f"{_malos} trials con L>T" + ("   <-- PROBLEMA" if _malos else "   OK"))
print(f"\nOBJETIVO LISTO ({len(WORD2ID)+1} clases).")

## 7. Entrenamiento

Un solo entrenamiento. Si hay OOM, baja el batch y sube `accum_grad` en la misma
proporcion, para que el batch efectivo no cambie y el resultado siga siendo
comparable con el barrido.

En Colab Pro+ puedes activar la ejecucion en segundo plano y cerrar el navegador.
El checkpoint se copia a Drive al terminar; sin eso muere con la VM y no se
podria ni volver a evaluar ni pasar el modelo de lenguaje.

In [ ]:
exp_dir   = f"{OUTPUT_DIR}/{TAG}";  os.makedirs(exp_dir, exist_ok=True)
drive_dir = f"{RESULTS_DIR}/{TAG}"; os.makedirs(drive_dir, exist_ok=True)

train_raw, val_raw, stats_dir, n_ses = get_datos(CFG)
os.makedirs(stats_dir, exist_ok=True)

data_info = make_data_info(CFG, aug=False)          # validacion: nunca aumentada
train_ds  = ez.dataset.ESPnetEZDataset(train_raw, data_info=make_data_info(CFG, aug=True))
valid_ds  = ez.dataset.ESPnetEZDataset(val_raw,   data_info=data_info)

set_encoder_lr_scale(CFG["enc_lr_scale"])
log_path = f"{exp_dir}/train.log"


def entrenar(batch_size):
    factor = max(1, CFG["batch_size"] // max(1, batch_size))
    c = {**CFG, "batch_size": batch_size, "accum_grad": CFG["accum_grad"] * factor}
    if factor > 1:
        print(f"  [OOM] batch {CFG['batch_size']}->{batch_size}, "
              f"accum_grad {CFG['accum_grad']}->{c['accum_grad']} (batch efectivo igual)")
    ft = build_finetune_config(c, stats_dir)
    pasos = max(1, len(train_raw) // max(1, batch_size))
    print(f"  batch={batch_size} · ~{pasos} pasos/epoca · "
          f"warmup {c['warmup']} (~{c['warmup']/pasos:.1f} epocas)")
    trainer = ez.Trainer(
        task="s2t", train_config=ft,
        train_dataset=train_ds, valid_dataset=valid_ds,
        build_model_fn=make_build_model_fn(c, n_ses), data_info=data_info,
        output_dir=exp_dir, stats_dir=stats_dir, ngpu=NGPU)
    asegurar_stats(trainer, stats_dir)
    train_raw.cerrar(); val_raw.cerrar()      # antes del fork: HDF5 no es fork-safe
    with CapturaLog(log_path):
        trainer.train()
    del trainer

print("=" * 78); print(f"[ENTRENANDO] {TAG}"); print("=" * 78)
t0 = time.time()
for intento, bs in enumerate([CFG["batch_size"], CFG["batch_size"] // 2,
                              max(1, CFG["batch_size"] // 4)]):
    try:
        entrenar(bs); break
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if intento == 2:
            raise
        print(f"  [OOM con batch={bs}] reintentando con {max(1, bs//2)}...")
MINUTOS = (time.time() - t0) / 60

# ── curva por epoca ───────────────────────────────────────────────────
df_ep = parse_train_log(log_path)
df_ep.to_csv(f"{drive_dir}/epocas.csv", index=False)
j = df_ep["valid_cer_ctc"].idxmin()
EPOCAS  = int(df_ep["epoch"].max())
MEJOR   = float(df_ep.loc[j, "valid_cer_ctc"])
EP_MEJOR = int(df_ep.loc[j, "epoch"])
ULT = df_ep.tail(10)
PENDIENTE = float(np.polyfit(ULT["epoch"], ULT["valid_cer_ctc"], 1)[0])

print(f"\n{EPOCAS} epocas en {MINUTOS/60:.1f} h · mejor valid cer_ctc {MEJOR:.3f} "
      f"en la epoca {EP_MEJOR}")
print(f"pendiente de las ultimas 10 epocas: {PENDIENTE:+.5f}/epoca "
      + ("(convergido)" if abs(PENDIENTE) < 1e-3 else "(AUN BAJANDO: 80 epocas se han quedado cortas)"))

if GUARDAR_CKPT:
    ckpt = find_best_checkpoint(exp_dir)
    shutil.copy(ckpt, f"{drive_dir}/{os.path.basename(ckpt)}")
    print(f"checkpoint -> {drive_dir}/{os.path.basename(ckpt)}")

## 8. Metricas CTC greedy

Decodificacion greedy —argmax por frame, colapsar repeticiones, quitar blanks—
sobre la **validacion completa**. No pasa por ningun decoder ni por ningun
modelo de lenguaje, asi que mide solo lo que el encoder extrae del sEEG.

Las referencias son `text_raw` normalizado en las tres unidades, con la misma
funcion de distancia, asi que **estas cifras si son comparables entre los tres
notebooks**. La `cer_ctc` del log de ESPnet no lo es: se calcula sobre la cadena
de tokens del objetivo, que es una unidad distinta en cada uno.

In [ ]:
# get_datos esta cacheado y SeeGDataset reabre los HDF5 solo, asi que val_raw
# sigue siendo utilizable aunque se cerrara antes del fork del DataLoader.
modelo = cargar_modelo_entrenado(CFG, exp_dir, n_ses)

print("evaluando con CTC greedy sobre la validacion completa...")
CER_MICRO, CER_G, WER_G = evaluar_ctc_greedy(CFG, modelo, val_raw)
SE_G = _ULTIMO_SE_GREEDY
print(f"\n  CER {CER_G:.3f} ± {1.96*SE_G:.3f} (IC95) · WER {WER_G:.3f}")
print(f"  cer_ctc micro (unidad del objetivo, NO comparable entre unidades): {CER_MICRO:.3f}")

## 9. Modelo de lenguaje

El n-grama se estima con **Kneser-Ney** (`lmplz`) sobre las transcripciones de
**train**. Usar las de validacion seria fuga.

pyctcdecode hace *fusion superficial dentro de la busqueda*: consulta el n-grama
en cada frontera de palabra mientras construye el beam, no reordena una lista ya
hecha. Por eso puede terminar en hipotesis que el beam puramente acustico nunca
habria generado, y por eso puede **superar al oraculo**. Si eso pasa, no es un
error: es la respuesta a PI3, porque significa que el LM no esta eligiendo entre
lo que el encoder propone, sino guiando lo que el encoder explora.

Se reportan cuatro filas:

- **greedy** — el encoder solo.
- **2-grama** y **3-grama** — con `alpha` y `beta` ajustados sobre los primeros
  200 trials y la cifra final sobre todos. Los dos salen de validacion: no hay
  test ciego, y eso va en las limitaciones.
- **oraculo** — el mejor WER alcanzable eligiendo dentro del beam acustico.

In [ ]:
import math, collections, subprocess
from itertools import groupby

try:
    import kenlm
    from pyctcdecode import build_ctcdecoder
except ImportError as _e:
    raise ImportError(f"Falta {_e.name}: se instala en la seccion 0.") from None

LMPLZ = "/content/kenlm/build/bin/lmplz"
HAY_LMPLZ = os.path.exists(LMPLZ)
print("Estimador de n-gramas:", "Kneser-Ney (lmplz)" if HAY_LMPLZ else "add-k (respaldo)")


def _vocab_de_arpa(ruta):
    """Palabras del bloque \1-grams:. pyctcdecode quiere PALABRAS, no <s>/<unk>."""
    vocab, dentro = [], False
    with open(ruta) as f:
        for linea in f:
            if linea.startswith("\\1-grams:"):
                dentro = True
                continue
            if dentro:
                if linea.startswith("\\") or not linea.strip():
                    break
                w = linea.split("\t")[1].strip()
                if not w.startswith("<"):
                    vocab.append(w)
    return vocab


def construir_arpa(frases, ruta, orden=2):
    """ARPA con Kneser-Ney interpolado. Solo con frases de TRAIN: usar las de
    validacion seria fuga."""
    if not HAY_LMPLZ:
        raise RuntimeError("lmplz no compilo; relanza la seccion 0")
    txt = ruta + ".txt"
    with open(txt, "w") as f:
        f.write("\n".join(frases) + "\n")
    # --discount_fallback es obligatorio: con ~8.000 frases faltan las
    # estadisticas de n-gramas que Kneser-Ney necesita y lmplz aborta sin el.
    subprocess.run([LMPLZ, "-o", str(orden), "--discount_fallback",
                    "--text", txt, "--arpa", ruta], check=True, capture_output=True)
    return ruta, _vocab_de_arpa(ruta)


def _barra(i, n, etiqueta, cada=None):
    cada = cada or max(1, n // 20)
    if (i + 1) % cada == 0 or i + 1 == n:
        print(f"\r    {etiqueta}: {i+1}/{n}", end="" if i + 1 < n else "\n", flush=True)


@torch.no_grad()
def logprobs_val(modelo, val_raw, n=None):
    """Log-probabilidades CTC de validacion, calculadas UNA sola vez.

    Ocupan poco (~35 KB por trial), asi que el barrido de alpha/beta y el
    oraculo no vuelven a tocar la GPU.
    """
    n = len(val_raw) if n is None else min(n, len(val_raw))
    idxs = np.linspace(0, len(val_raw) - 1, n).astype(int)
    lps, refs = [], []
    for j, i in enumerate(idxs):
        d = val_raw[int(i)]
        sp = torch.tensor(preparar_speech(d, CFG), dtype=torch.float32,
                          device=DEVICE).unsqueeze(0)
        ln = torch.tensor([sp.shape[1]], dtype=torch.long, device=DEVICE)
        enc, _ = modelo.encode(sp, ln)
        if isinstance(enc, tuple):
            enc = enc[0]
        lps.append(torch.log_softmax(modelo.ctc.ctc_lo(enc)[0].float(), -1).cpu().numpy())
        refs.append(d["text_raw"])
        _barra(j, n, "codificando")
    return lps, refs


def greedy_desde_lp(lp):
    """El MISMO greedy que evaluar_ctc_greedy: argmax, colapso, quitar blanks.

    Con pyctcdecode y beam_width=1 el resultado NO es identico, y entonces las
    cifras no cuadrarian con las de la seccion anterior.
    """
    obj = objetivo_info(CFG)
    ids = [k for k, _ in groupby(lp.argmax(-1).tolist()) if k != obj["blank"]]
    dtxt = obj.get("destok_texto")
    return dtxt(ids) if dtxt else obj["destok"](ids)


def medir(hips, refs):
    return (float(np.mean([cer(r, h) for r, h in zip(refs, hips)])),
            float(np.mean([wer(r, h) for r, h in zip(refs, hips)])))


def wer_oraculo(dec_sin_lm, lps, refs, beam=100, a_texto=None):
    """Cota inferior: el mejor WER alcanzable eligiendo DENTRO del beam acustico.

    Se usa el decodificador SIN modelo de lenguaje a proposito. Asi el beam
    contiene lo que propone el encoder por si solo, y el oraculo responde a:
    con un LM perfecto que solo reordenase esa lista, ¿hasta donde llegaria?
    La distancia entre el LM real y el oraculo es el margen que le queda al LM
    reordenando; si el LM real SUPERA al oraculo, es que no esta reordenando,
    esta cambiando la busqueda.
    """
    cer_o = wer_o = n_hip = 0.0
    for i, (lp, r) in enumerate(zip(lps, refs)):
        # pyctcdecode poda por defecto con token_min_logp=-5 y beam_prune_logp=-10.
        # Para un oraculo eso es demasiado estricto: interesa la lista mas amplia
        # que el encoder considere plausible, no la mas limpia.
        hips = [h[0] for h in dec_sin_lm.decode_beams(
            lp, beam_width=beam, token_min_logp=-8.0, beam_prune_logp=-20.0)]
        if a_texto:
            hips = [a_texto(h) for h in hips]
        n_hip += len(hips)
        wers = [wer(r, h) for h in hips]
        j = int(np.argmin(wers))
        wer_o += wers[j]
        cer_o += cer(r, hips[j])
        _barra(i, len(lps), "oraculo")
    n = len(lps)
    return cer_o / n, wer_o / n, n_hip / n

# ── Etiquetas para pyctcdecode ────────────────────────────────────────
# pyctcdecode entra en modo subpalabra si alguna etiqueta empieza por "▁", y
# entonces cada "▁xxx" abre palabra nueva. Prefijando TODAS las etiquetas, cada
# clase del CTC es una palabra completa y el n-grama puntua exactamente las
# transiciones entre palabras. Es fusion dentro de la busqueda, igual que en
# char, no reordenacion.
#
# OJO CON LA INTERPRETACION: el lexico ya esta en la capa de salida, asi que el
# LM aqui solo aporta el orden de las palabras, no su forma. Se espera una
# ganancia mucho menor que en char, y ESO es el resultado: mide cuanto del
# hueco de WER entre char y word era ortografia y cuanto era sintaxis.
#
# pyctcdecode avisa "UNK token ▁⁇▁ not found": busca el simbolo de desconocido de
# sentencepiece, que aqui no existe. Comprobado que es inofensivo.
ETIQUETAS = [""] + ["▁" + ID2WORD[i] for i in range(1, len(WORD2ID) + 1)]
A_TEXTO = None


def decodificar(dec, lp, beam=100):
    return dec.decode(lp, beam_width=beam)

In [ ]:
BEAM_LM   = 100
N_BARRIDO = 200                       # trials para ajustar alpha y beta
ALPHAS    = (0.3, 0.6, 1.0)
BETAS     = (0.0, 1.5, 3.0)

frases = [_norm(t) for t in train_raw.textos()]       # SOLO train: usar val seria fuga
lps, refs = logprobs_val(modelo, val_raw, n=None)
N = len(refs)
print(f"\n  {N} trials de validacion · LM sobre {len(frases)} frases de train\n")

dec_sin = build_ctcdecoder(ETIQUETAS)                 # beam puramente acustico

filas = []
cer_g, wer_g = medir([greedy_desde_lp(lp) for lp in lps], refs)
filas.append(("greedy", "", "", cer_g, wer_g))
print(f"  greedy: CER {cer_g:.3f} · WER {wer_g:.3f}")

hips_por_orden = {}
for orden in (2, 3):
    ruta, vocab = construir_arpa(frases, f"/content/lm{orden}.arpa", orden=orden)
    print(f"\n  {orden}-grama · {len(vocab)} palabras · "
          f"ARPA {os.path.getsize(ruta)/1e6:.1f} MB")

    mejor = None
    lote = lps[:min(N_BARRIDO, N)]
    for a in ALPHAS:
        for b in BETAS:
            dec = build_ctcdecoder(ETIQUETAS, kenlm_model_path=ruta,
                                   unigrams=vocab, alpha=a, beta=b)
            hs = [dec.decode(lp, beam_width=BEAM_LM) for lp in lote]
            w = float(np.mean([wer(r, h) for r, h in zip(refs[:len(lote)], hs)]))
            print(f"    alpha={a:<4} beta={b:<4} WER={w:.3f}  (n={len(lote)})")
            if mejor is None or w < mejor[0]:
                mejor = (w, a, b)

    _, a, b = mejor
    dec = build_ctcdecoder(ETIQUETAS, kenlm_model_path=ruta,
                           unigrams=vocab, alpha=a, beta=b)
    hips = []
    for i, lp in enumerate(lps):
        hips.append(dec.decode(lp, beam_width=BEAM_LM))
        _barra(i, N, f"{orden}-grama final")
    c, w = medir(hips, refs)
    hips_por_orden[orden] = hips
    filas.append((f"{orden}-grama", a, b, c, w))
    print(f"  {orden}-grama: CER {c:.3f} · WER {w:.3f}  (alpha={a}, beta={b})")

cer_o, wer_o, n_hip = wer_oraculo(dec_sin, lps, refs, beam=BEAM_LM, a_texto=A_TEXTO)
filas.append(("oraculo", "", "", cer_o, wer_o))
print(f"  oraculo: CER {cer_o:.3f} · WER {wer_o:.3f}  ({n_hip:.0f} hipotesis/trial)")

## 10. Tabla final y curva

Todo lo que va a la memoria, en un sitio. Se guarda en Drive.

In [ ]:
import matplotlib.pyplot as plt

print(f"\n  {'':12s} {'CER':>8} {'WER':>8}   ({N} trials)")
for nom, a, b, c, w in filas:
    print(f"  {nom:12s} {c:8.3f} {w:8.3f}" + (f"   alpha={a}, beta={b}" if a != "" else ""))

print("\n  ── ejemplos ──")
for k in range(3):
    print(f"\n  REF    : {_norm(refs[k])}")
    print(f"  greedy : {greedy_desde_lp(lps[k])}")
    for orden in (2, 3):
        print(f"  {orden}-grama: {hips_por_orden[orden][k]}")

# ── CSV para la memoria ───────────────────────────────────────────────
tabla = pd.DataFrame(filas, columns=["metodo", "alpha", "beta", "cer", "wer"])
tabla.insert(0, "unidad", UNIDAD)
tabla["n_trials"] = N
tabla.to_csv(f"{drive_dir}/metricas_lm.csv", index=False)

resumen = pd.DataFrame([dict(
    unidad=UNIDAD, tag=TAG, clases=objetivo_info(CFG)["vocab"],
    epocas=EPOCAS, epoca_mejor=EP_MEJOR, pendiente_final=round(PENDIENTE, 5),
    min_valid_cer_ctc=round(MEJOR, 4),
    cer_greedy=round(CER_G, 4), wer_greedy=round(WER_G, 4),
    cer_greedy_ic95=round(1.96 * SE_G, 4),
    cer_lm=round(min(f[3] for f in filas[1:3]), 4),
    wer_lm=round(min(f[4] for f in filas[1:3]), 4),
    cer_oraculo=round(cer_o, 4), wer_oraculo=round(wer_o, 4),
    horas=round(MINUTOS / 60, 2), semilla=CFG["seed"])])
resumen.to_csv(f"{drive_dir}/resumen.csv", index=False)
display(resumen.T)

# ── curva ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_ep["epoch"], df_ep["valid_cer_ctc"], label="valid cer_ctc")
if "train_cer_ctc" in df_ep.columns:
    ax.plot(df_ep["epoch"], df_ep["train_cer_ctc"], ls="--", alpha=.6, label="train cer_ctc")
ax.axvline(EP_MEJOR, c="crimson", ls=":", lw=1)
ax.annotate(f"mejor: {MEJOR:.3f} (ep {EP_MEJOR})", (EP_MEJOR, MEJOR),
            textcoords="offset points", xytext=(8, 10), color="crimson", fontsize=9)
ax.set_xlabel("epoca"); ax.set_ylabel("cer_ctc")
ax.set_title(f"best_{UNIDAD}: curva de entrenamiento")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.savefig(f"{drive_dir}/curva.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nGuardado en {drive_dir}/  (resumen.csv · metricas_lm.csv · epocas.csv · curva.png)")

---

## Notas

**Orden de ejecucion.** Las celdas dependen unas de otras. Ejecuta de arriba
abajo, y despues de la seccion 0 **reinicia el entorno** y vuelve a empezar
desde la seccion 1.

**Que es comparable y que no.**

- `cer_greedy` y `wer_greedy` **si** son comparables entre los tres notebooks:
  misma referencia (`text_raw` normalizado), misma distancia de edicion.
- `min_valid_cer_ctc` (la del log de ESPnet) **no** lo es: se calcula sobre la
  cadena de tokens del objetivo, y un token es un caracter, una palabra o un
  fonema segun el notebook.
- El WER de `best_word` tiene un suelo estructural por las palabras OOV, que la
  seccion 6 mide. Hay que reportarlo al lado de la cifra.
- En `best_fon` el LM **reordena** una lista n-best; en los otros dos **guia la
  busqueda**. El efecto del LM no es comparable entre fon y los otros dos.

**Limitaciones que van en la memoria.**

- `alpha` y `beta` se ajustan sobre validacion y la cifra final se da sobre
  validacion. No hay test ciego.
- La semilla es fija (2024). La dispersion entre semillas medida en el barrido
  es de ±0.011 de CER, mayor que el IC95 del tamano de validacion (±0.007): la
  incertidumbre que manda es la de semilla.